In [8]:

# Restoran Şube Başarı Analizi  
import pandas as pd
import numpy as np

df = pd.read_csv("../01-data/restaurant_branch_success.csv")
print("Boyut:", df.shape) #Veri hakkında kısa ilk bakış
df.head()

Boyut: (2000, 6)


,ticket_id,model,lokasyon_tipi,musteri_memnun,siparis_suresi_dk,masa_sayisi
0,T0001,Sube_B,avm,evet,11,25
1,T0002,Sube_A,sanayi,evet,10,8
2,T0003,Sube_B,avm,hayir,9,40
3,T0004,Sube_A,sanayi,hayir,12,20
4,T0005,Sube_B,avm,evet,8,37


In [9]:
#Eksik veri var mı kontrol
df.info()
print("Eksik değer kaç adet", df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ticket_id          2000 non-null   object
 1   model              2000 non-null   object
 2   lokasyon_tipi      2000 non-null   object
 3   musteri_memnun     2000 non-null   object
 4   siparis_suresi_dk  2000 non-null   int64 
 5   masa_sayisi        2000 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 93.9+ KB
Eksik değer kaç adet ticket_id            0
model                0
lokasyon_tipi        0
musteri_memnun       0
siparis_suresi_dk    0
masa_sayisi          0
dtype: int64


In [7]:
# Her Şube İçin Genel Memnuniyet Oranı

# musteri_memnun evet/hayırı --> oran hesabı için 1/0 çevir
df["memnun_bool"] = (df["musteri_memnun"] == "evet").astype(int) 

# şubeye göre grupla: toplam anket, memnun sayısı, oran
sube_memnuniyet = df.groupby("model").agg(
    anket_sayisi = ("ticket_id", "count"),
    memnun_sayisi = ("memnun_bool", "sum"),
    memnuniyet_orani = ("memnun_bool", "mean")
).reset_index()

# oranı yüzdeye çevir
sube_memnuniyet["memnuniyet_yuzde"] = (sube_memnuniyet["memnuniyet_orani"] * 100).round(2)

sube_memnuniyet


,model,anket_sayisi,memnun_sayisi,memnuniyet_orani,memnuniyet_yuzde
0,Sube_A,1000,539,0.539,53.9
1,Sube_B,1000,774,0.774,77.4


In [4]:
import pandas as pd

df = pd.read_csv("../01-data/restaurant_branch_success.csv")
df.head()

,ticket_id,model,lokasyon_tipi,musteri_memnun,siparis_suresi_dk,masa_sayisi
0,T0001,Sube_B,avm,evet,11,25
1,T0002,Sube_A,sanayi,evet,10,8
2,T0003,Sube_B,avm,hayir,9,40
3,T0004,Sube_A,sanayi,hayir,12,20
4,T0005,Sube_B,avm,evet,8,37


In [8]:
# Her Şube İçin Lokasyon Tipine Göre Memnuniyet Oranı

df["memnun_bool"] = (df["musteri_memnun"] == "evet").astype(int)

# şube ve lokasyon ikilisine göre grupla
sube_lokasyon_memnuniyet = df.groupby(["model", "lokasyon_tipi"]).agg(
    anket_sayisi = ("ticket_id", "count"),
    memnun_sayisi = ("memnun_bool", "sum"),
    memnuniyet_orani = ("memnun_bool", "mean")
).reset_index()

# oranı yüzdeye çevir (2 ondalık)
sube_lokasyon_memnuniyet["memnuniyet_yuzde"] = (
    sube_lokasyon_memnuniyet["memnuniyet_orani"] * 100
).round(2)

sube_lokasyon_memnuniyet

,model,lokasyon_tipi,anket_sayisi,memnun_sayisi,memnuniyet_orani,memnuniyet_yuzde
0,Sube_A,avm,200,185,0.9250,92.50
1,Sube_A,sanayi,800,354,0.4425,44.25
2,Sube_B,avm,800,690,0.8625,86.25
3,Sube_B,sanayi,200,84,0.4200,42.00


In [12]:
# Ham frekans: her şubeye kaç avm ve kaç sanayi anketi gelmiş
dagilim = pd.crosstab(
    df["model"],
    df["lokasyon_tipi"],
    margins=True,
    margins_name="Toplam"
)
dagilim

lokasyon_tipi,avm,sanayi,Toplam
model,,,
Sube_A,200,800,1000
Sube_B,800,200,1000
Toplam,1000,1000,2000


In [15]:
# Lokasyon tipinin yönetici fark etmeksizin memnuniyet üzerindeki etkisi, avm müşterisi sanayi müşterisinden 44 puan daha yüksek memnuniyet veriyor.lokasyon değişiminin gücü
df.groupby("lokasyon_tipi")["memnun_bool"].mean().mul(100).round(2)

lokasyon_tipi
avm       87.5
sanayi    43.8
Name: memnun_bool, dtype: float64

In [18]:
# Her şubenin kendi içinde lokasyon dağılımı (%)
dagilim_pct = (
    pd.crosstab(df["model"], df["lokasyon_tipi"], normalize="index") * 100
).round(2)
dagilim_pct

lokasyon_tipi,avm,sanayi
model,,
Sube_A,20.0,80.0
Sube_B,80.0,20.0


In [ ]:
# Her Şubeye Atanan Lokasyon Tipi Dağılımı

# Şube × lokasyon kombinasyonuna göre anket sayısı
sube_lokasyon_dagilim = df.groupby(["model", "lokasyon_tipi"]).size().reset_index(name="anket_sayisi")

# Her şubenin kendi içindeki yüzde payı
sube_lokasyon_dagilim["yuzde"] = (
    sube_lokasyon_dagilim.groupby("model")["anket_sayisi"]
    .transform(lambda x: x / x.sum() * 100)
    .round(2)
)

sube_lokasyon_dagilim

,model,lokasyon_tipi,anket_sayisi,yuzde
0,Sube_A,avm,200,20.0
1,Sube_A,sanayi,800,80.0
2,Sube_B,avm,800,80.0
3,Sube_B,sanayi,200,20.0
